In [6]:
from rdkit import Chem

import numpy as np
import xgboost
from xgboost import XGBClassifier
import sys
sys.path.append('molecule-web-app/backend')
from ml_helper import make_fingerprint # pyright: ignore[reportMissingImports]
sys.path.append('molecule-web-app/backend/rules')  # replace with your local path
# Now you can import your module as usual
from rules import execute_step, run_next_step
import helper
from next_step import follow_on
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') 


In [7]:
rxn_tests = [("CC(=O)CC(=O)C.[OH-]","CC(=O)C=C(C)[O-].O"),
              ("CC(=O)CC(=O)C.N","CC(=O)C=C(C)[O-].[NH4+]"),
              ("CC(=O)CC(=O)C.CC(C)(C)[O-]","CC(=O)C=C(C)[O-].CC(C)(C)O"),
              ("CC(O)C=C(C)[O-]", "CC=C[C+](C)[O-].[OH-]"),
              ("CC(=O)[CH-]C(C)O", "CC(O)C=C(C)[O-]"),
              ("CC(O)C=C(C)[O-]", "CC(=O)[CH-]C(C)O"),
              ("CC(=O)[CH-]C(C)O","CC=CC(C)=O.[OH-]"),
              ("CC(=[OH+])C.C=CO", "CC(C)(O)C[CH+]O"),
              ("CC=[OH+].C=CO", "CC(O)C[CH+]O"),
              ("CC(=O)C.C=C[O-]", "CC(C)([O-])C[CH+][O-]"),
              ("CC(=O)C.C=C[O-]", "CC(C)=O.[CH2-]C=O"),
              ("CC=O.C=C[O-]", "CC([O-])C[CH+][O-]"),
              ("CC=O.C=C[O-]", "CC=O.[CH2-]C=O"),
              ("CC(=O)C=C.N", "CC(=O)[CH-]C[NH3+]"),
              ("CC(=O)C=C.O", "CC(=O)[CH-]C[OH2+]"),
              ("CC(=O)C=C.CO", "CC(=O)[CH-]C[OH+]C")
              ]

In [10]:
clf = xgboost.XGBClassifier()
clf.load_model('models/v53/xg_model.json')
for rxn in rxn_tests:
    found = False
    reactant_smiles, product_smiles = rxn
    reactant_mol = Chem.MolFromSmiles(reactant_smiles)
    prod_mol = Chem.MolFromSmiles(product_smiles)
    prod_inchi = Chem.MolToInchi(prod_mol, options="/FixedH")
    product_mols, mechs, mol_inp  = run_next_step("Placeholder", True, True, reactant_mol)
    for i, product_mol in enumerate(product_mols):
        fp = make_fingerprint(reactant_mol, mechs[i])
        pred = clf.predict(np.array([fp]))[0]
        if pred == 1:
            if Chem.MolToInchi(product_mol, options="/FixedH") == prod_inchi:
                found = True
    if not found:
        print(f"Failed to find product for reaction: {reactant_smiles} -> {product_smiles}")


Failed to find product for reaction: CC(=O)CC(=O)C.CC(C)(C)[O-] -> CC(=O)C=C(C)[O-].CC(C)(C)O
Failed to find product for reaction: CC(O)C=C(C)[O-] -> CC=C[C+](C)[O-].[OH-]
Failed to find product for reaction: CC(O)C=C(C)[O-] -> CC(=O)[CH-]C(C)O
Failed to find product for reaction: CC(=O)[CH-]C(C)O -> CC=CC(C)=O.[OH-]
